### RAG Pipelines- Data Ingestion to Vector Database Pipelines


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def process_all_pdfs(directory_path):
    """Process all PDF files in the given directory."""
    
    all_documents = []
    pdf_dir = Path(directory_path)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

        print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents


all_pdf_documents = process_all_pdfs("../data/pdf")

In [ ]:
all_pdf_documents

In [ ]:
### Text splitting get into chuncks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks for good performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", " ", ""]
        )
    split_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} documents from {len(split_docs)} chuncks")

    # Show Example of a chunk
    if split_docs:
        print("\nExample of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}..")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [ ]:
chunks=split_documents(all_pdf_documents)

### Embedding and VectorStoreDB

In [ ]:
import uuid
from typing import Any, Dict, List, Tuple

import chromadb
from chromadb.config import Settings
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.

        Args:
            model_name (str): HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading Sentence Transformer model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts:List[str])-> np.ndarray:
        """Generate embeddings for a list of texts using the Sentence Transformer model

        Args:
        texts : List of texts strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), wmbedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded. Please call _load_model() first.")

        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts,batch_size=64,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

### VectorStore



In [ ]:
import os
import uuid
from typing import List
import numpy as np
import chromadb
# Assuming LangChain's Document schema or similar
from langchain_core.documents import Document  


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """Initialize the vector store

        Args:
            collection_name (str): Name of the ChromaDB collection
            persist_directory (str): Path to directory where vector store will be persisted
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"},
            )
            print(
                f"Vector store initialized successfully. Collection: {self.collection_name}"
            )
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:  # Fixed indentation here
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store

        Args:
            documents (List[Document]): List of Document objects to add
            embeddings (np.ndarray): Embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"Adding {len(documents)} documents to vector store")

        # Fixed comment syntax here (# instead of @)
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID (Fixed uuid.uuid4 typo here)
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_texts.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts,
            )
            print(
                f"Successfully added {len(documents)} documents to vector store"
            )
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Instantiation placed outside the class scope
if __name__ == "__main__":
    vectorstore = VectorStore()